# **Tập dữ liệu**

## Load Data

In [1]:
from datasets import load_dataset

ds = load_dataset(
    "imagefolder",
    data_dir="../data/raw/intel_images",
)

/home/kvgkvg/.conda/envs/datatest/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


EmptyDatasetError: The directory at /home/kvgkvg/Documents/university/DM/dmaa-lab1/data/raw/intel_images doesn't contain any data files

In [ ]:
print(ds)

## Chuẩn bị dữ liệu

Mặc dù tập dữ liệu đã được chia sẵn thành train và test set, nhưng do yêu cầu của bài tập tập trung vào bước tiền xử lý và số lượng ảnh trong dataset khá lớn khiến thời gian tính toán bị kéo dài. Vì vậy, trong phạm vi bài tập này, nhóm chỉ chọn ngẫu nhiên 5.000 ảnh từ tập train để tiến hành xử lý và đánh giá.

In [ ]:
# Define constants
N = 5000
H = 150
W = 150
SEED = 42

In [ ]:
dataset = ds["train"].shuffle(seed=SEED).select(range(N))
len(dataset)

In [ ]:
dataset[0]

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image

image_list = []
label_list = []
original_size = (150, 150)

for item in tqdm(dataset):
    pil_img = item["image"].convert("RGB")
    img_resized_pil = pil_img.resize(original_size, Image.LANCZOS)
    img_resized_np = np.array(img_resized_pil)

    image_list.append(img_resized_np)
    label_list.append(item["label"])

In [ ]:
images = np.array(image_list)
labels = np.array(label_list)

In [ ]:
print(f"\nShape of images: {images.shape}")  # (5000, 150, 150, 3) ~ (N, H, W, 3)
print(f"Shape of label: {labels.shape}")  # (5000,)

In [ ]:
print(label_list)

In [ ]:
class_names = dataset.features["label"].names
print(f"Classes: {class_names}")
print(f"Number of classes: {len(class_names)}")
unique_labels = np.sort(pd.Series(label_list).unique())
print(f"Labels: {unique_labels}")

---
# **Phân tích thống kê tập dữ liệu**

## Tính và trực quan hóa phân phối giá trị pixel trên toàn tập (histogram, KDE) theo từng kênh màu.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

In [ ]:
def plot_pixel_dist_kde(image_set, sample_size=None):
    colors = ("red", "green", "blue")
    plt.figure(figsize=(12, 5))

    for i, color in enumerate(colors):
        # dùng ravel để lấy mảng 1D array của từng kênh màu (5000 x 150 x 150, )
        all_channel_vals = image_set[:, :, :, i].ravel()
        total_pixels = len(all_channel_vals)
        print(f"Kích thước channel {color}: {total_pixels}")

        if (
            sample_size and total_pixels > sample_size
        ):  # Downsampling khi kích thước mảng quá lớn
            step = total_pixels // sample_size
            channel_vals = all_channel_vals[::step][:sample_size]
            print(f"Sample {len(channel_vals)} pixel cho channel {color.upper()}.")
        else:
            channel_vals = all_channel_vals
            print(f"Dùng tất cả {total_pixels} pixels cho channel {color.upper()}.")

        sns.kdeplot(channel_vals, color=color, label=f"{color.upper()} channel")

    plt.title("KDE Plot phân phối giá trị Pixel trên toàn tập dữ liệu")
    plt.xlabel("Giá trị Pixel (0-255)")
    plt.legend()
    plt.show()


plot_pixel_dist_kde(image_set=images, sample_size=1000000)

In [ ]:
def plot_pixel_dist_histogram(image_set):
    colors = ("red", "green", "blue")
    plt.figure(figsize=(12, 6))

    for i, color in enumerate(colors):
        channel_vals = image_set[:, :, :, i].ravel()
        plt.hist(
            channel_vals,
            bins=256,
            color=color,
            alpha=0.2,
            label=f"{color.upper()} channel",
        )

    plt.title("Histogram phân phối giá trị Pixel trên toàn tập dữ liệu")
    plt.xlabel("Giá trị Pixel (0-255)")
    plt.ylabel("Tần suất")
    plt.legend()
    plt.grid(axis="y", alpha=0.5)
    plt.show()


plot_pixel_dist_histogram(image_set=images)

## Phân tích mất cân bằng lớp (class imbalance): tính tỉ lệ mỗi lớp, kiểm tra xem có lớp nào chiếm tỉ lệ vượt mức 3x so với lớp ít nhất không.


In [ ]:
label_info = pd.DataFrame(
    {
        "label": np.sort(pd.Series(label_list).unique()),
        "counts": pd.Series(label_list).value_counts().sort_index(),
        "class_names": class_names,
    }
)
label_info

In [ ]:
def analyze_class_imbalance(label_info):
    min_count = label_info["counts"].min()
    max_count = label_info["counts"].max()
    imbalance_ratio = max_count / min_count

    plt.figure(figsize=(10, 5))
    sns.barplot(
        x=label_info["label"],
        y=label_info["counts"],
        hue=label_info["label"].index,
        palette="viridis",
        legend=False,
    )

    # Vẽ đường ngưỡng 3x
    plt.axhline(
        y=min_count * 3,
        color="red",
        linestyle="--",
        label=f"Ngưỡng 3x lớp ít nhất ({min_count*3})",
    )

    plt.title("Phân phối số lượng mẫu theo lớp")
    plt.ylabel("Số lượng ảnh")
    plt.xticks()
    plt.legend()
    plt.show()

    print(f"\nTỉ lệ Max/Min: {imbalance_ratio:.2f}")

    if imbalance_ratio > 3:
        print(
            f"Tập dữ liệu mất cân bằng (Lớp nhiều nhất gấp {imbalance_ratio:.2f} lần lớp ít nhất)."
        )
    else:
        print(f"Tập dữ liệu cân bằng (Tỉ lệ {imbalance_ratio:.2f} < 3x).")


analyze_class_imbalance(label_info)

## Phát hiện ảnh trùng lặp hoặc gần trùng bằng hàm băm perceptual hash (pHash). Báo cáo tỉ lệ trùng lặp và xử lý chúng.

In [ ]:
import imagehash
from PIL import Image

In [ ]:
def detect_dup_img_phash(images, labels, class_names):
    hashes = {}
    duplicates = []  # các index bị trùng
    dup_img = []  # lưu các ảnh bị trùng đê hiển thị

    print("Tính pHash cho từng ảnh...")
    for i in tqdm(range(len(images))):
        img_pil = Image.fromarray(images[i])
        h = str(imagehash.phash(img_pil))

        if h in hashes:
            duplicates.append(i)
            dup_img.append((hashes[h], i))
        else:
            hashes[h] = i

    total_imgs = len(images)
    dup_count = len(duplicates)
    dup_rate = (dup_count / total_imgs) * 100

    print(f"Tổng số ảnh kiểm tra: {total_imgs}")
    print(f"Số lượng ảnh trùng: {dup_count}")
    print(f"Tỉ lệ trùng lặp: {dup_rate:.2f}%")

    if dup_img:
        for orig_idx, dup_idx in dup_img:
            fig, ax = plt.subplots(1, 2, figsize=(8, 4))
            ax[0].imshow(images[orig_idx])
            ax[0].set_title(f"Index: {orig_idx}\nLớp: {class_names[labels[orig_idx]]}")
            ax[1].imshow(images[dup_idx])
            ax[1].set_title(f"Index: {dup_idx}\nLớp: {class_names[labels[dup_idx]]}")
            plt.show()

    return duplicates


duplicate_indices = detect_dup_img_phash(images, labels, class_names)

In [ ]:
if len(duplicate_indices) > 0:
    # Loại bỏ các hàng tương ứng với index trùng
    images_clean = np.delete(images, duplicate_indices, axis=0)
    labels_clean = np.delete(labels, duplicate_indices, axis=0)

    print(f"Kích thước mảng ban đầu: {images.shape}")
    print(f"Kích thước mảng sau khi lọc ảnh trùng: {images_clean.shape}")

    images = images_clean
    labels = labels_clean
else:
    pass

## Phân tích độ tương phản và độ sáng toàn cục: tính mean intensity và standard deviation theo lớp, thể hiện qua boxplot phân lớp.


In [ ]:
def boxplot_brightness_contrast(image_set, labels, names):
    means = np.mean(image_set, axis=(1, 2, 3))
    stds = np.std(image_set, axis=(1, 2, 3))

    df = pd.DataFrame(
        {
            "Độ sáng toàn cục": means,
            "Độ tương phản": stds,
            "Lớp": [names[l] for l in labels],
        }
    )

    fig, ax = plt.subplots(1, 2, figsize=(16, 6))
    sns.boxplot(x="Lớp", y="Độ sáng toàn cục", data=df, ax=ax[0])
    ax[0].set_title("Độ sáng toàn cục theo Lớp")

    sns.boxplot(x="Lớp", y="Độ tương phản", data=df, ax=ax[1])
    ax[1].set_title("Độ tương phản theo Lớp")
    plt.show()

    return df


df_brightness_contrast = boxplot_brightness_contrast(images, labels, class_names)

In [ ]:
df_brightness_contrast.head()